In [1]:
import boto3
import json
import dotenv
import os
dotenv.load_dotenv()

region = "ap-southeast-1"
s3 = boto3.client("s3", region_name=region)
iam = boto3.client("iam", region_name=region)

### Create Policy

In [2]:
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:GetObjectVersion"
            ],
            "Resource": "arn:aws:s3:::zomato-pipeline-s3/*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "s3:ListBucket",
                "s3:GetBucketLocation"
            ],
            "Resource": "arn:aws:s3:::zomato-pipeline-s3"
        }
    ]
}

response = iam.create_policy(
    PolicyName="ZomatoS3ReadOnlyPolicy",
    PolicyDocument=json.dumps(policy_document),
    Description="Allow read access to zomato S3 bucket"
)

print("Policy ARN:", response["Policy"]["Arn"])

Policy ARN: arn:aws:iam::678835429553:policy/ZomatoS3ReadOnlyPolicy


### Create Role

In [7]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "ec2.amazonaws.com" # Dummy value will replace with snowflake service principal
            },
            "Action": "sts:AssumeRole",
            "Condition": {}
        }
    ]
}

response = iam.create_role(
    RoleName="SnowflakeS3Role",
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="Role for Snowflake to access S3"
)

role_arn = response["Role"]["Arn"]
print("Role ARN:", role_arn)

Role ARN: arn:aws:iam::678835429553:role/SnowflakeS3Role


### Get snowflake Principal trust and externalID then update IAMRole
`snowflake/002_s3_integration.sql`

In [4]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": os.getenv("STORAGE_AWS_IAM_USER_ARN")
            },
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "sts:ExternalId": os.getenv("STORAGE_AWS_EXTERNAL_ID")
                }
            }
        }
    ]
}

response = iam.update_assume_role_policy(
    RoleName="SnowflakeS3Role",
    PolicyDocument=json.dumps(trust_policy)
)

print("Trust policy updated successfully")

Trust policy updated successfully


### Attach Policies

In [3]:
iam.attach_role_policy(
    RoleName="SnowflakeS3Role",
    PolicyArn="arn:aws:iam::678835429553:policy/ZomatoS3ReadOnlyPolicy"
)

print("Policy attached successfully")

Policy attached successfully
